In [7]:
import pylast
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Initialize
network = pylast.LastFMNetwork(
    api_key='61c031a558849b580efcc91ff7f526ac',
    api_secret='db7519713f221f1f33d910544c1ff7e7',
    username='TinkuJiya'
)


# Step 1: Create training dataset
def create_training_data(username, n_liked=50, n_disliked=50):
    """
    Create binary classification dataset:
    X = artist features
    y = 1 (user likes) or 0 (user dislikes)
    """
    
    user = network.get_user("TinkuJiya")
    
    # Get liked artists (user's top artists)
    top_tracks = user.get_top_tracks(limit=n_liked)
    liked_artists = list(set([track.item.artist.name for track in top_tracks]))
    
    # Get disliked artists (random artists not in top)
    disliked_artists = []
    visited = set(liked_artists)
    
    for artist_name in liked_artists[:10]:
        artist_obj = network.get_artist(artist_name)
        similar = artist_obj.get_similar()
        
        for sim in similar:
            if sim.item.name not in visited and len(disliked_artists) < n_disliked:
                disliked_artists.append(sim.item.name)
                visited.add(sim.item.name)
    
    # Extract features for all artists
    X_data = []
    y_data = []
    
    # Liked artists (label = 1)
    for artist_name in liked_artists[:n_liked]:
        try:
            features = extract_artist_features(artist_name)
            if features:
                X_data.append(features)
                y_data.append(1)
        except:
            pass
    
    # Disliked artists (label = 0)
    for artist_name in disliked_artists[:n_disliked]:
        try:
            features = extract_artist_features(artist_name)
            if features:
                X_data.append(features)
                y_data.append(0)
        except:
            pass
    
    return np.array(X_data), np.array(y_data)

# Step 2: Extract features for artist
def extract_artist_features(artist_name):
    """Extract 8 numerical features for artist"""
    try:
        artist = network.get_artist(artist_name)
        
        top_tags = artist.get_top_tags()
        tags_list = [float(tag.weight) for tag in top_tags[:5]]
        
        # Pad if less than 5 tags
        while len(tags_list) < 5:
            tags_list.append(0)
        
        # Features:
        # 0-4: Top 5 tag weights
        # 5: Listener count (normalized log)
        # 6: Playcount (normalized log)
        # 7: Number of top tags
        
        listeners = artist.get_listener_count()
        playcount = artist.get_playcount()
        n_tags = len(top_tags)
        
        features = tags_list + [
            np.log1p(listeners) / 10,  # Normalize
            np.log1p(playcount) / 10,
            n_tags / 50  # Normalize (max ~50 tags)
        ]
        
        return features
    except:
        return None

# Step 3: Train Random Forest
def train_random_forest(username):
    """Train RF classifier on user's music preference"""
    
    X, y = create_training_data(username)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train model
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X_train_scaled, y_train)
    
    # Evaluate
    train_score = rf.score(X_train_scaled, y_train)
    test_score = rf.score(X_test_scaled, y_test)
    
    print(f"Train Accuracy: {train_score:.3f}")
    print(f"Test Accuracy: {test_score:.3f}")
    print(f"Feature Importance: {rf.feature_importances_}")
    
    return rf, scaler

# Step 4: Recommend using model
def recommend_rf(username , candidate_artists, rf, scaler, n_recommendations=10):
    """Score candidate artists using trained model"""
    
    predictions = {}
    
    for artist_name in candidate_artists:
        try:
            features = extract_artist_features(artist_name)
            if features:
                features_scaled = scaler.transform([features])
                probability = rf.predict_proba(features_scaled)[0][1]  # P(like)
                predictions[artist_name] = probability
        except:
            pass
    
    # Sort by predicted probability
    top_recs = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    
    return top_recs

# Usage
rf, scaler = train_random_forest('TinkuJiya')
candidates = ['Radiohead', 'Coldplay', 'Muse', 'Arctic Monkeys', 'The Strokes']
recs = recommend_rf('TinkuJiya', candidates, rf, scaler)

for artist, prob in recs:
    print(f"{artist}: {prob:.3f} (probability of liking)")

print(extract_artist_features("The Strokes"))

Train Accuracy: 1.000
Test Accuracy: 0.529
Feature Importance: [0.         0.12798559 0.19396586 0.13394679 0.11564395 0.20122613
 0.17499622 0.05223547]
The Strokes: 0.780 (probability of liking)
Coldplay: 0.680 (probability of liking)
Arctic Monkeys: 0.660 (probability of liking)
Muse: 0.640 (probability of liking)
Radiohead: 0.201 (probability of liking)
[100.0, 85.0, 53.0, 40.0, 19.0, np.float64(1.5581141805441407), np.float64(2.0045745618892843), 0.2]
